In [1]:
import math
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data})"
    
    def __radd__(self, other):
        return self + other
    
    def __add__(self, other):
        if not isinstance(other, Value): 
            other = Value(other)
        out = Value(self.data + other.data, (self, other), _op='+')

        def _backward():
            self.grad += 1.0*out.grad
            other.grad += 1.0*out.grad

        out._backward = _backward
        return out
    
    def __mul__(self, other): 
        if not isinstance(other, Value): 
            other = Value(other)
        out = Value(self.data*other.data, (self, other), _op='*')

        def _backward():
            self.grad += other.data*out.grad
            other.grad += self.data*out.grad

        out._backward = _backward
        return out
    
    def __rmul__(self, other):
        return self*other
    
    def __neg__(self):
        return self*-1
    
    def __sub__(self, other):
        return self + (-other)
    
    def __rsub__(self, other):
        return -(self - other)
    
    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supportint int/float powers for now"
        out = Value(self.data**other, (self, ), f'**{other}')

        def _backward():
            self.grad += other*(self.data**(other - 1))*out.grad

        out._backward = _backward
        return out
        
    def __truediv__(self, other):
        return self * (other**-1)
    
    def exp(self):
        out = Value(math.exp(self.data), (self, ), _op='exp')

        def _backward():
            self.grad += out.data*out.grad
        out._backward = _backward

        return out
    
    def tanh(self):
        x = self.data
        val = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
        out = Value(val, (self, ), _op='tanh')

        def _backward():
            self.grad += (1 - out.data**2)*out.grad 
        out._backward = _backward

        return out
    
    def backward(self):
        queue = []
        visited = set()

        def topo(now):
            if now not in visited:
                visited.add(now)
                for next in now._prev:
                    topo(next)
                queue.append(now)

        topo(self)
        for node in queue:
            node.grad = 0.0
        self.grad = 1.0
        for node in reversed(queue):
            node._backward()

a = Value(2.5)
print(1 - a)
print(a - 1)

Value(data=-1.5)
Value(data=1.5)


In [2]:
import torch
x1 = torch.Tensor([2.0]).double()
x1.requires_grad = True
x2 = torch.Tensor([0.0]).double()
x2.requires_grad = True
w1 = torch.Tensor([-3.0]).double()
w1.requires_grad = True
w2 = torch.Tensor([1.0]).double()
w2.requires_grad = True
b = torch.Tensor([6.8813735870195432]).double()
b.requires_grad = True
n = x1*w1 + x2*w2 + b
o = torch.tanh(n)

print(o.data.item())
o.backward()
print('---')
print('x2', x2.grad.item())
print('w2', w2.grad.item())
print('x1', x1.grad.item())
print('w1', w1.grad.item())

0.7071066904050358
---
x2 0.5000001283844369
w2 0.0
x1 -1.5000003851533106
w1 1.0000002567688737


In [11]:
import random

class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(0)

    def __call__(self, x):
        act = sum((w*x for w, x in zip(self.w, x)), self.b)
        out = act.tanh()
        return out

    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        out = [n(x) for n in self.neurons]
        return out[0] if len(out) == 1 else out

    def parameters(self):
        params = []
        for neuron in self.neurons:
            params.extend(neuron.parameters())
        return params

class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(sz) - 1)]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        params = []
        for layer in self.layers:
            params.extend(layer.parameters())
        return params

x = [2.0, 3.0, -1.0]
n = MLP(3, [4, 4, 1])
n(x)


Value(data=0.21110431644599129)

In [23]:
xs = [[2.0, 3.0, -1.0],
      [3.0, -1.0, 0.5],
      [0.5, 1.0, 1.0],
      [1.0, 1.0, -1.0]] 
ys = [1.0, -1.0, -1.0, -1.0]
for _ in range(10):
    ypred = [n(x) for x in xs]
    loss = sum((ypred_i - y_i)**2 for ypred_i, y_i in zip(ypred, ys))
    print(_ , loss.data)
    for p in n.parameters():
        p.grad = 0.0
    loss.backward()
    for p in n.parameters():
        p.data += -0.1*p.grad

print([n(x) for x in xs])

0 0.014572649343292662
1 0.014261287819007984
2 0.013961520954048755
3 0.013672746271945656
4 0.013394401055360328
5 0.01312595918457335
6 0.012866928268808883
7 0.012616847039540613
8 0.012375282978557276
9 0.012141830156727158
[Value(data=0.9277434031383345), Value(data=-0.9751904219762255), Value(data=-0.9462108404662191), Value(data=-0.9435526561558231)]
